# Final Dataset Selection & ML Preparation Specification

This notebook closes the **dataset-analysis phase**.

It consumes the completed CIC-IDS2017/CIC-IDS2018 CSV artifacts and converts the evidence into an explicit, version-controlled specification for the final ML pipeline.

**Primary decision:** CIC-IDS2017 is the working primary dataset. CIC-IDS2018 remains the secondary benchmark/reference dataset.

This notebook does **not** load the raw CIC datasets and does not modify any data.

In [2]:
from pathlib import Path
import zipfile, shutil, json, re
import pandas as pd
import numpy as np
from google.colab import files

UPLOAD_ROOT = Path("/content/upload")
WORK_ROOT = Path("/content")
UPLOAD_ROOT.mkdir(exist_ok=True)
uploaded = files.upload()

zips = [Path("/content")/n for n in uploaded if n.lower().endswith(".zip")]
if not zips:
    raise FileNotFoundError("Upload the result-artifact ZIP.")
zip_path = zips[0]

with zipfile.ZipFile(zip_path) as z:
    z.extractall("/content")

def locate(name):
    candidates = [
        Path("/content/results")/name,
        Path("/content")/name,
    ]
    for p in candidates:
        if p.exists():
            return p
    return None

RESULTS_2017 = locate("cicids2017")
RESULTS_2018 = locate("cicids2018")
CROSS_RESULTS = locate("cross_dataset_comparison")

print("2017:", RESULTS_2017)
print("2018:", RESULTS_2018)
print("Cross-dataset:", CROSS_RESULTS)

if RESULTS_2017 is None or RESULTS_2018 is None:
    raise FileNotFoundError("Could not locate both CIC-IDS2017 and CIC-IDS2018 result trees.")

Saving notebook13_artifacts.zip to notebook13_artifacts.zip
2017: /content/cicids2017
2018: /content/cicids2018
Cross-dataset: None


## 1. Inventory the evidence

The inventory is intentionally conservative: if a metric is unavailable in an artifact, it is not reconstructed by guessing.

In [3]:
def inventory(root, dataset):
    rows=[]
    for p in sorted(root.rglob("*.csv")):
        rows.append({
            "dataset": dataset,
            "relative_path": str(p.relative_to(root)),
            "file": p.name,
            "size_bytes": p.stat().st_size
        })
    return pd.DataFrame(rows)

inventory_df = pd.concat([
    inventory(RESULTS_2017, "CIC-IDS2017"),
    inventory(RESULTS_2018, "CIC-IDS2018")
], ignore_index=True)

display(inventory_df)

""


## 2. Decision statement

The cross-dataset work established the following engineering conclusion:

- CIC-IDS2017 is substantially smaller and therefore materially easier to process and train on.
- CIC-IDS2017 has a much smaller missing/infinite-value burden in the completed analysis.
- CIC-IDS2018 provides broader scale/class coverage but imposes substantially greater computational and preprocessing cost.
- The project therefore proceeds with **CIC-IDS2017 as the primary ML dataset**.
- CIC-IDS2018 is retained as a secondary comparative dataset and is not silently discarded.

In [4]:
decision = pd.DataFrame([
    ["Primary dataset", "CIC-IDS2017", "Best overall engineering trade-off for the intended ML IDS under available compute."],
    ["Secondary dataset", "CIC-IDS2018", "Retained for comparative evidence and possible future benchmarking."],
    ["Raw data modification in this notebook", "None", "Selection only; actual transformation is performed in Notebook 14."],
], columns=["decision_item","selection","justification"])
display(decision)

,decision_item,selection,justification
0,Primary dataset,CIC-IDS2017,Best overall engineering trade-off for the int...
1,Secondary dataset,CIC-IDS2018,Retained for comparative evidence and possible...
2,Raw data modification in this notebook,None,Selection only; actual transformation is perfo...


## 3. Final preprocessing guidelines

1. Normalize column-name whitespace consistently.
2. Treat `Label` as the target and never allow it into feature transformations.
3. Replace positive/negative infinity with missing values.
4. Handle missing numeric values using parameters fitted on the training split only.
5. Remove features explicitly rejected by the completed preprocessing analysis.
6. Remove constant predictors.
7. Preserve the target labels before any optional binary/multiclass transformation.
8. Fit scaling parameters on training data only.
9. Never perform oversampling/undersampling before the train/test boundary.
10. Persist the feature list, label mapping, imputer/scaler and split manifest.
11. Keep the original raw CSV files untouched.
12. Preserve a provenance record linking every transformation to the prior analysis artifacts.

In [5]:
FINAL_SPEC = {
    "dataset": "CIC-IDS2017",
    "target_column": "Label",
    "label_mode": "multiclass",
    "random_state": 42,
    "replace_infinite_with_nan": True,
    "imputation": "median_numeric_fit_on_train_only",
    "scaling": "StandardScaler_fit_on_train_only",
    "remove_constant_features": True,
    "remove_duplicates": True,
    "oversampling": False,
    "undersampling": False,
    "primary_split": "stratified train/validation/test",
    "split_ratios": {"train": 0.70, "validation": 0.15, "test": 0.15},
    "raw_data_modified": False
}
spec_dir = Path("/content/results/final_dataset_selection")
spec_dir.mkdir(parents=True, exist_ok=True)

pd.DataFrame([
    {"parameter": k, "value": json.dumps(v) if isinstance(v,(dict,list)) else v}
    for k,v in FINAL_SPEC.items()
]).to_csv(spec_dir/"preprocessing_specification.csv", index=False)

pd.DataFrame([
    {"feature": "Label", "role": "target", "reason": "CIC flow attack/benign class label"},
]).to_csv(spec_dir/"target_specification.csv", index=False)

decision.to_csv(spec_dir/"final_dataset_decision.csv", index=False)
print("Specification written to:", spec_dir)

Specification written to: /content/results/final_dataset_selection


## 4. Evaluation design

Because the EDA showed strong class imbalance and capture-session concentration, the final ML evaluation must report:

- macro F1
- weighted F1
- macro recall
- balanced accuracy
- per-class precision/recall/F1
- confusion matrix
- class support
- training/inference time
- model size where practical

Accuracy may be reported, but it is **not sufficient as the primary metric**.

The evaluation should also include a leakage check and a documented distinction between the ordinary stratified split and any session/file-aware validation experiment.

In [6]:
evaluation_spec = pd.DataFrame([
    ["Primary", "Macro F1", "Treats classes equally; important under severe imbalance."],
    ["Primary", "Macro Recall", "Measures whether minority attack classes are detected."],
    ["Primary", "Balanced Accuracy", "Reduces dominance of majority class."],
    ["Primary", "Per-class F1", "Exposes failure on rare attacks."],
    ["Secondary", "Weighted F1", "Useful overall summary while retaining support weighting."],
    ["Secondary", "Accuracy", "Context only because BENIGN dominates."],
    ["Diagnostic", "Confusion Matrix", "Shows attack-to-attack and attack/benign confusion."],
    ["Diagnostic", "Training/Inference Time", "Relevant to practical deployment."],
], columns=["priority","metric","purpose"])
display(evaluation_spec)
evaluation_spec.to_csv(spec_dir/"evaluation_specification.csv", index=False)

,priority,metric,purpose
0,Primary,Macro F1,Treats classes equally; important under severe...
1,Primary,Macro Recall,Measures whether minority attack classes are d...
2,Primary,Balanced Accuracy,Reduces dominance of majority class.
3,Primary,Per-class F1,Exposes failure on rare attacks.
4,Secondary,Weighted F1,Useful overall summary while retaining support...
5,Secondary,Accuracy,Context only because BENIGN dominates.
6,Diagnostic,Confusion Matrix,Shows attack-to-attack and attack/benign confu...
7,Diagnostic,Training/Inference Time,Relevant to practical deployment.


## Conclusion

The evaluation framework defined in this notebook prioritizes metrics that remain informative under the severe class imbalance present in intrusion-detection datasets. **Macro F1, Macro Recall, Balanced Accuracy, and Per-class F1** are therefore treated as the primary evaluation metrics, as they give equal importance to attack categories and expose failures on minority or rare attack classes.

**Weighted F1** and **Accuracy** are retained as secondary metrics. Although they provide useful overall performance context, they should not be used as the principal basis for model selection because the dominance of the BENIGN class can make aggregate performance appear stronger than the actual detection capability for minority attack classes.

The **confusion matrix** and **training/inference time** serve as diagnostic measures. The confusion matrix provides additional insight into attack-to-attack and attack/benign misclassification patterns, while computational time provides an indication of the practical feasibility of the resulting model.

Consequently, subsequent model comparisons should primarily favor models that demonstrate strong and consistent **Macro F1, Macro Recall, Balanced Accuracy, and per-class F1**, particularly on minority attack categories, rather than models that achieve high accuracy through majority-class performance alone. This evaluation strategy provides a more reliable basis for assessing the effectiveness and practical usefulness of the final intrusion-detection models.